In [3]:
import json
import argparse
import functools
import itertools
from datetime import datetime

In [4]:
import torch
import numpy as np 
import pandas as pd
from rich.live import Live
from rich.panel import Panel

from fintorch.deep.model import *
from fintorch.deep.module import create_module_from_args
from fintorch.deep.transform.feature import *
from fintorch.deep.transform.label import *
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.enum import TimeFrame
from fintorch.deep.metrics import Metrics
from fintorch.setting import RICH_PROGRESS_COLUMNS
from fintorch.utils.timestamp import create_timestamps
from fintorch.utils.args import DefaultArgumentParser

SyntaxError: invalid syntax (exchange.py, line 17)

In [5]:
string_args = [
    "--start-date", "2023-06-01",
    "--stop-date", "2024-04-01",
    
    "--num-hidden-layers", "2",
    
    "--shuffle", "True",
    "--val-length", "30",
    "--test-length", "30",
    "--train-length", "300",

    "--batch-size", "64",
    "--epochs-count", "50",
    
    "--lr", "0.01",
    "--dropout", "0.1",
    "--weight-decay", "0.001",
]

args = DefaultArgumentParser.parse(args=string_args)

print(args.__dict__)

NameError: name 'DefaultArgumentParser' is not defined

In [6]:
module = create_module_from_args(
    feature_transform_type=PreviousFractalsFeatureTransform,
    label_transform_type=ForwardMiddleSmaLabelTransform,
    model_type=LSTM,
    args=args
)

module.time_frame

NameError: name 'args' is not defined

# Load data collection and optimize module

In [7]:
dc = ONLINE_EXCHANGE.future.data.get_data_collection(symbols=[module.symbol], time_frames=[module.time_frame])

with Live(refresh_per_second=2) as live, module:
    for status in module.optimize(dc=dc, start_date=args.start_date, stop_date=args.stop_date):
        live.update(Panel.fit(str(status), title=str(module)))

NameError: name 'ONLINE_EXCHANGE' is not defined

In [ ]:
timestamps = module.get_timestamps(dc=dc, start_date=args.start_date, stop_date=args.stop_date)

# create y and y_hat values
with module:
    sf_generator = module.dataset.label_transform.transform_sf(dc=dc, timestamps=timestamps)
    y_array = np.concatenate([sf for sf in sf_generator if sf is not None])
    timestamps = timestamps[:len(y_array)]
    y_hat_dict = module.predict(dc=dc, timestamps=timestamps, mode="val")

# convert y and y_hat values to torch.Tensor
y = torch.from_numpy(y_array)
y_hat = torch.from_numpy(np.array(list(y_hat_dict.values())))

print(np.isnan(y).sum())
print(np.isnan(y_hat).sum())
print()

# calculate metrics
metrics = Metrics(criterion=module.trainer.criterion, y=y, y_hat=y_hat)

# print("Loss:        {:.4f}".format(metrics.objective))
print("Accuracy:    {:.2f} {:.2f}".format(metrics.accuracy, metrics.probability_accuracy))
print("Recall-UP:   {:.2f} Precision-UP:   {:.2f}".format(metrics.recall(label=1), metrics.precision(label=1)))
print("Recall-Down: {:.2f} Precision-Down: {:.2f}".format(metrics.recall(label=0), metrics.precision(label=0)))

# Draw perdictions plot

In [ ]:
with module:
    module.show_ohlc_plot(dc=dc, start_date=args.start_date, stop_date=args.stop_date, mode="val")

In [ ]:
df = dc.get_candles_df(symbol=module.symbol, time_frame=module.time_frame).copy()
df.insert(0, "datetime", [datetime.fromtimestamp(ts) for ts in df.index])
        
prediction_dict = {k: np.argmax(v) if not np.isnan(v).max() else np.nan for k, v in y_hat_dict.items()}
df["prediction"] = pd.Series(prediction_dict)
df["side"] = np.where(1 == df.prediction, 1, -1)
df["roc"] = df.close.shift(-10) / df.open - 1
df["pnl"] = df.roc * df.side
df = df.dropna()

print(df.prediction.value_counts())
print(df.side.value_counts())
print(df.pnl.mean())
print((0 < df.pnl).mean())